In [3]:
#!pip install lingua-language-detector

In [ ]:
from pathlib import Path
import pickle 
import pandas as pd
from lingua import LanguageDetectorBuilder, Language
from tqdm import tqdm
from preprocess import preprocess_text

tqdm.pandas()
DATA_DIR = Path("../../data")

In [ ]:
detector = LanguageDetectorBuilder.from_all_languages().build()

def detect_language(x):
    text = preprocess_text(x[1])
    language = detector.detect_language_of(text)
    return text, language

df = pd.read_csv(DATA_DIR/"MetaHate.tsv", sep='\t')
df[['clean_text', 'language']] = df.progress_apply(detect_language, axis=1, result_type='expand')

In [ ]:
df.to_csv("MetaHate_languages.csv", sep='\t')
removed_df = df[df['language'] != Language.ENGLISH]
removed_df.to_csv('foreign_comments.csv', sep='\t')

In [ ]:
with open("processed.pkl", "wb") as f:
    df['language'] = df['language'].apply(str)
    removed_df['language'] = removed_df['language'].apply(str)
    pickle.dump((df, removed_df), f)

In [ ]:
with open("processed.pkl", "rb") as f:
    df, removed_df = pickle.load(f)
    df['language'] = df['language'].apply(lambda x: eval(x))
    removed_df['language'] = removed_df['language'].apply(lambda x: eval(x))

In [ ]:
range_balkans_subset = list(range(379410, 390154))
filter_empty = removed_df['clean_text'].apply(lambda x: True if len(x.strip()) == 0 else len(x.split()) == 1)
filter_balkans_subset = removed_df.index.isin(range_balkans_subset)
LANGUAGES_IGNORED = [Language.ARABIC, Language.PERSIAN, Language.GEORGIAN, Language.GREEK, 
                     Language.CHINESE, Language.HEBREW, Language.JAPANESE, Language.ARMENIAN,
                     Language.THAI, Language.VIETNAMESE, Language.MONGOLIAN, Language.URDU,
                     Language.HINDI, Language.MARATHI, Language.PUNJABI, Language.TAMIL]
index_ignored = removed_df[removed_df['language'].isin(LANGUAGES_IGNORED) & ~filter_empty & ~filter_balkans_subset].index
index_turkish = removed_df[(removed_df['language'] == Language.TURKISH) & ~filter_empty & ~filter_balkans_subset].index
index_balkans = removed_df[removed_df['language'].isin([Language.BOSNIAN, Language.CROATIAN, Language.SERBIAN]) & ~filter_empty & ~filter_balkans_subset].index
LANGUAGES_SLAVIC = [Language.RUSSIAN, Language.BELARUSIAN, Language.UKRAINIAN, Language.BULGARIAN]
index_slavic = removed_df[removed_df['language'].isin(LANGUAGES_SLAVIC) & ~filter_empty & ~filter_balkans_subset].index
index_lith = removed_df[(removed_df['language'] == Language.LITHUANIAN) & ~filter_empty & ~filter_balkans_subset].index
index_polish = removed_df[(removed_df['language'] == Language.POLISH) & ~filter_empty & ~filter_balkans_subset].index
LANGUAGES_SPANISH = [Language.SPANISH, Language.PORTUGUESE, Language.BASQUE, Language.CATALAN]
index_spanish = removed_df[removed_df['language'].isin(LANGUAGES_SPANISH) & ~filter_empty & ~filter_balkans_subset].index
index_no_lang = removed_df['language'].isnull()
# Mostly noise
index_swahili = removed_df[(removed_df['language'] == Language.SWAHILI) & ~filter_empty & ~filter_balkans_subset].index
index_german = removed_df[(removed_df['language'] == Language.GERMAN) & ~filter_empty & ~filter_balkans_subset].index
index_hungarian = removed_df[(removed_df['language'] == Language.HUNGARIAN) & ~filter_empty & ~filter_balkans_subset].index
redundant_index = (
    range_balkans_subset + 
    index_ignored.tolist() + 
    index_no_lang[index_no_lang].index.tolist() +     
    filter_empty.index[filter_empty].tolist() + 
    index_turkish[index_turkish > 1000000].tolist() +
    index_balkans[index_balkans > 380000].tolist() + 
    index_slavic.tolist() + 
    index_lith[index_lith > 1000000].tolist() +
    index_polish[index_polish > 1000000].tolist() + 
    index_swahili[index_swahili > 1000000].tolist() + 
    index_german[index_german > 1000000].tolist() + 
    index_hungarian[index_hungarian > 1000000].tolist() + 
    index_spanish.to_list()
)
print(len(set(redundant_index)))

19647


In [ ]:
removed_df[(removed_df['language'] == Language.HUNGARIAN) & ~filter_empty & ~filter_balkans_subset]

,label,text,clean_text,language
3192,1,@FrankieJGrande omg gtfo white faggot,omg gtfo white faggot,Language.HUNGARIAN
8703,0,Dese niggaz on Dat guala shit....... http://t....,Dese niggaz on Dat guala shit,Language.HUNGARIAN
12189,1,Kanye west is a faggot,Kanye west is a faggot,Language.HUNGARIAN
19077,1,RT @harmon_lauren: Jenna's a faggot,RT Jenna's a faggot,Language.HUNGARIAN
67466,0,The latest TheSékondi Daily! https://t.co/nwer...,The latest TheSékondi Daily! Thanks to,Language.HUNGARIAN
...,...,...,...,...
1091709,0,== Prakrit Sattak ==,Prakrit Sattak,Language.HUNGARIAN
1095072,0,one else deletes it.),one else deletes it,Language.HUNGARIAN
1096225,0,Ha igen akkor szivesen atforditanek ezt azt.,Ha igen akkor szivesen atforditanek ezt azt,Language.HUNGARIAN
1097405,0,"Még nem tudtam, mert akkor épp egyetemen volta...","Még nem tudtam, mert akkor épp egyetemen volta...",Language.HUNGARIAN


In [ ]:
removed_df

,label,text,clean_text,language
85,1,"""@Blackman38Tide: @WhaleLookyHere @HowdyDowdy1...","queer"" gaywad",Language.TAGALOG
87,0,"""@BrosConfessions: This bitch was so ungratefu...",This bitch was so ungrateful fr.. LULWHORE,Language.SOTHO
89,1,"""@CB_Baby24: @white_thunduh alsarabsss"" hes a ...","alsarabsss"" hes a beaner smh you can tell hes ...",Language.WELSH
121,0,"""@Ferocious_Ghost: @1stName_Bravo Aw."" ...fag,...","Aw. "".. fag, don't tweet ""aw"" to me lol",Language.TAGALOG
220,0,"""@RachieSoul: @DizzleOfficial inna di ghetto!!...","inna di ghetto!! "" You alone lmaoo",Language.ITALIAN
...,...,...,...,...
1100988,0,Për Ilirët jemi marr vesh tani. Mos u brengos ...,Për Ilirët jemi marr vesh tani. Mos u brengos ...,Language.ALBANIAN
1100996,0,OMFG WTF OMFG WTF OMFG WTF OMFG WTF OMFG WTF O...,OMFG WTF OMFG WTF OMFG WTF OMFG WTF OMFG WTF O...,Language.NYNORSK
1101021,0,"Mein lieber Brendel, Ich bin auch deutscher He...","Mein lieber Brendel, Ich bin auch deutscher He...",Language.GERMAN
1101064,1,NIGEL IS A CRAZY IDIOT!!!,NIGEL IS A CRAZY IDIOT,Language.POLISH


In [ ]:
with open('excluded.csv', 'w') as f:
    f.write('\n'.join(map(str, sorted(set(redundant_index)))))

Combine both excluded by semantic deduplication and cleaning indices:

In [2]:
index_removed = pd.read_csv("removed.csv", header=None)[0].tolist()
index_excluded = pd.read_csv("excluded.csv", header=None)[0].tolist()
index_combined = list(set(index_removed + index_excluded))
with open('removed_all.csv', 'w') as f:
    f.write('\n'.join(map(str, sorted(index_combined))))